# Sistema RAG — Asistente Conversacional Clínico

**TFM:** Sistema de Apoyo a la Decisión Clínica en Oncología Pediátrica
**Autor:** Alonso Castañón González

**Objetivo de este notebook:** Construir el sistema RAG (Retrieval-Augmented Generation) que permite a oncólogos y familias consultar en lenguaje natural información sobre osteosarcoma y sarcoma de Ewing pediátrico, combinando conocimiento general con la predicción específica del modelo XGBoost entrenado en `03_modelo_ML.ipynb`.

**Arquitectura:** LangGraph con 3 nodos (`retrieve` → `postfiltering` → `generate`), embeddings locales gratuitos (HuggingFace) e indexado en Qdrant (Docker local), con generación mediante la API de OpenAI.

In [21]:
import os
import pickle
import json
import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from qdrant_client import QdrantClient
import requests
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from collections import Counter
from langchain_qdrant import QdrantVectorStore
from qdrant_client.http.models import Distance, VectorParams
from typing import TypedDict
from langchain_core.messages import convert_to_openai_messages
from langgraph.graph import StateGraph, END, START
from typing import Optional

## 1) Setup

Se cargan las credenciales desde `.env` y se verifica la conexión a Qdrant, HuggingFace y OpenAI antes de empezar.

In [2]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY is not None, "No se encontró OPENAI_API_KEY en el archivo .env"

print("Variables de entorno cargadas correctamente")

QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

qdrant_client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

print("Colecciones existentes en Qdrant:", qdrant_client.get_collections())

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={'device': 'cpu'}
)

test_embedding = embeddings.embed_query("prueba de conexión")
print(f"Embedding generado localmente, dimensión: {len(test_embedding)}")

llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)

test_response = llm.invoke("Responde solo con la palabra 'OK' si me recibes.")
print(f"Respuesta del LLM: {test_response.content}")

Variables de entorno cargadas correctamente
Colecciones existentes en Qdrant: collections=[CollectionDescription(name='tfm_oncologia_pediatrica')]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding generado localmente, dimensión: 384
Respuesta del LLM: OK


## 2) Documentos y metadatos

Se descargan los 8 documentos NCI mediante scraping, extrayendo el contenido principal de cada página (recortando el pie de página administrativo común a las fichas PDQ: permisos, citación, información de ensayos clínicos) y etiquetando cada uno por enfermedad y tipo. Estos documentos serán los que alimenten al RAG y de los cuales surgirán sus respuestas. Es por eso que se han buscado documentos de varios tipos, más específicos sobre los tratamientos del osteosarcoma y de ewing, o más generales de cara a la explicación a las familias. 

In [3]:
import requests
from bs4 import BeautifulSoup

# Encabezados a partir de los cuales se recorta el contenido
MARKERS = [
    "About This PDQ Summary", "About PDQ", "Purpose of This Summary",
    "Reviewers and Updates", "Clinical Trial Information",
    "Permission to Use This Summary", "Disclaimer", "Contact Us",
    "Related resources"
]

def scrape_nci_page(url):
    res = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    res.raise_for_status()
    soup = BeautifulSoup(res.text, "html.parser")

    main = soup.find("main") or soup.find(id="main-content")
    if main is None:
        raise ValueError(f"No se encontró el contenido principal en {url}")

    title = soup.find("h1").get_text(strip=True) if soup.find("h1") else url

    parts = []
    for el in main.find_all(["h2", "h3", "h4", "p", "li"]):
        text = el.get_text(strip=True)
        if not text:
            continue
        # Solo cortar en encabezados reales, no en enlaces del índice "On This Page"
        if el.name in ["h2", "h3", "h4"] and any(marker in text for marker in MARKERS):
            break
        parts.append(text)

    return {"title": title, "url": url, "text": "\n".join(parts)}


documentos_nci = [
    {"url": "https://www.cancer.gov/types/bone/patient/osteosarcoma-treatment-pdq",
     "enfermedad": "osteosarcoma", "tipo": "tratamiento"},
    {"url": "https://www.cancer.gov/types/bone/patient/ewing-treatment-pdq",
     "enfermedad": "ewing", "tipo": "tratamiento"},
    {"url": "https://www.cancer.gov/about-cancer/treatment/types/chemotherapy",
     "enfermedad": "general", "tipo": "tratamiento"},
    {"url": "https://www.cancer.gov/about-cancer/treatment/types/surgery",
     "enfermedad": "general", "tipo": "tratamiento"},
    {"url": "https://www.cancer.gov/about-cancer/treatment/types/radiation-therapy",
     "enfermedad": "general", "tipo": "tratamiento"},
    {"url": "https://www.cancer.gov/about-cancer/coping/caregiver-support/parents",
     "enfermedad": "general", "tipo": "apoyo_emocional"},
    {"url": "https://www.cancer.gov/about-cancer/understanding/what-is-cancer",
 "enfermedad": "general", "tipo": "conceptos_basicos"},
    {"url": "https://www.cancer.gov/types/childhood-cancers",
     "enfermedad": "general", "tipo": "informativo"},
]

In [4]:
documentos = []
for doc in documentos_nci:
    scraped = scrape_nci_page(doc["url"])
    scraped["enfermedad"] = doc["enfermedad"]
    scraped["tipo"] = doc["tipo"]
    documentos.append(scraped)
    print(f"OK - {scraped['title']} — {len(scraped['text'])} caracteres")

OK - Osteosarcoma Treatment (PDQ®)–Patient Version — 24655 caracteres
OK - Ewing Sarcoma Treatment (PDQ®)–Patient Version — 27690 caracteres
OK - Chemotherapy to Treat Cancer — 10991 caracteres
OK - Surgery to Treat Cancer — 15139 caracteres
OK - Radiation Therapy to Treat Cancer — 10886 caracteres
OK - Support for Families: Childhood Cancer — 24036 caracteres
OK - What Is Cancer? — 18242 caracteres
OK - Childhood Cancers — 9558 caracteres


## 3) Chunking

Se divide cada documento en fragmentos manejables para el retrieval. Se usa `chunk_size=1000`, ya que en principio los documentos NCI son narrativos y se benefician de fragmentos algo más largos para mantener contexto coherente También con `chunk_overlap=200` y conservando los metadatos (`enfermedad`, `tipo`, `title`, `url`) en cada fragmento.

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

split_docs = []
for doc in documentos:
    chunks = splitter.split_text(doc["text"])
    for chunk in chunks:
        if chunk.strip():
            split_docs.append(
                Document(
                    page_content=chunk,
                    metadata={
                        "title": doc["title"],
                        "url": doc["url"],
                        "enfermedad": doc["enfermedad"],
                        "tipo": doc["tipo"],
                    }
                )
            )

print(f"Total de fragmentos generados: {len(split_docs)}")

conteo = Counter(d.metadata["title"] for d in split_docs)
for titulo, n in conteo.items():
    print(f"  {titulo}: {n} fragmentos")

Total de fragmentos generados: 183
  Osteosarcoma Treatment (PDQ®)–Patient Version: 33 fragmentos
  Ewing Sarcoma Treatment (PDQ®)–Patient Version: 36 fragmentos
  Chemotherapy to Treat Cancer: 15 fragmentos
  Surgery to Treat Cancer: 19 fragmentos
  Radiation Therapy to Treat Cancer: 14 fragmentos
  Support for Families: Childhood Cancer: 30 fragmentos
  What Is Cancer?: 23 fragmentos
  Childhood Cancers: 13 fragmentos


## 4) Indexado en Qdrant

Se crea la colección en Qdrant (recreándola si ya existiera, para evitar duplicados al reejecutar el notebook) y se suben los 183 fragmentos con sus embeddings y metadatos.

In [6]:
COLLECTION_NAME = "tfm_oncologia_pediatrica"

# Recrear la colección para evitar duplicados
if qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.delete_collection(COLLECTION_NAME)

qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)  # 384 = dimensión de MiniLM-L6-v2
)

qdrant = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)

qdrant.add_documents(split_docs)

print(f"Colección '{COLLECTION_NAME}' creada con {len(split_docs)} fragmentos indexados")

Colección 'tfm_oncologia_pediatrica' creada con 183 fragmentos indexados


In [7]:
resultados_prueba = qdrant.similarity_search("¿Qué es la quimioterapia neoadyuvante?", k=3)

for r in resultados_prueba:
    print(f"[{r.metadata['enfermedad']} / {r.metadata['tipo']}] {r.metadata['title']}")
    print(r.page_content[:200], "...\n")

[osteosarcoma / tratamiento] Osteosarcoma Treatment (PDQ®)–Patient Version
Chemotherapy
Chemotherapy (also called chemo) uses drugs to stop the growth of cancer cells. Chemotherapy either kills the cancer cells or stops them from dividing.   Chemotherapy may be given alone o ...

[general / tratamiento] Chemotherapy to Treat Cancer
Chemotherapy works against cancer by killing fast-growing cancer cells.
Credit: National Cancer Institute
Chemotherapy(also called chemo) is a type of cancer treatment that uses drugs to kill cancerce ...

[osteosarcoma / tratamiento] Osteosarcoma Treatment (PDQ®)–Patient Version
For tumors  that have recurred twice, treatment may include:surgery to remove the cancer and chemotherapychemotherapy alone
surgery to remove the cancer and chemotherapy
chemotherapy alone
Learn more  ...



## 5) Cadena RAG (LangGraph)

Se construye el grafo de 3 nodos: `retrieve`, que busca `k=8` fragmentos candidatos en Qdrant, `postfiltering`, donde el LLM valora uno a uno si cada fragmento es relevante a la pregunta descartando así el posible ruido, y `generate`, que redacta la respuesta usando solo los fragmentos ya filtrados. `k=8` se ajusta al tamaño de 183 fragmentos totales, ya que un valor mayor es desproporcionado para este volumen.

In [10]:
class RAGState(TypedDict):
    question: str
    docs: list
    answer: str


def retrieve_node(state: RAGState) -> RAGState:
    docs = qdrant.similarity_search(state['question'], k=8)
    state["docs"] = docs
    return state


def postfiltering_node(state: RAGState) -> RAGState:
    final_docs = []
    for doc in state['docs']:
        inputs = {"messages": [
            ("system", f"""
            sobre este documento:
            ----------
            {doc.page_content}
            --------------
            """),
            ("human", 
            f"""{state['question']}, dime con la etiqueta [RELEVANT] si el documento es relevante para la pregunta o [NOT RELEVANT] si el contenido no tiene nada que ver.""")
        ]}
        messages_langchain = convert_to_openai_messages(inputs['messages'])
        resp = llm.invoke(messages_langchain).content
        if "[RELEVANT]" in resp:
            final_docs.append(doc)
    state['docs'] = final_docs
    return state


def generate_node(state: RAGState) -> RAGState:
    inputs = {"messages": [
        ("system", f"""
        Eres un asistente que responde preguntas sobre osteosarcoma y sarcoma de Ewing
        pediátrico, basándote únicamente en estos documentos:
        ----------
        {state['docs']}
        --------------
        """),
        ("human", f"""{state['question']}
        1. Responde en español, de forma clara.
        2. Si no ves documentos relevantes para la pregunta, dilo explícitamente en vez de inventar información.""")
    ]}
    messages_langchain = convert_to_openai_messages(inputs['messages'])
    resp = llm.invoke(messages_langchain).content
    state['answer'] = resp
    return state

In [11]:
graph = StateGraph(RAGState)

graph.add_node("retrieve", retrieve_node)
graph.add_node("postfiltering", postfiltering_node)
graph.add_node("generate", generate_node)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "postfiltering")
graph.add_edge("postfiltering", "generate")
graph.add_edge("generate", END)

rag = graph.compile()

print("Grafo RAG construido correctamente")

Grafo RAG construido correctamente


In [12]:
resultado = rag.invoke({"question": "¿Qué es la quimioterapia neoadyuvante?"})
print(resultado["answer"])

La quimioterapia neoadyuvante es un tipo de tratamiento que se utiliza antes de la cirugía o la radioterapia. Su objetivo es reducir el tamaño de un tumor para facilitar su extirpación o para mejorar la efectividad de otros tratamientos. Sin embargo, no tengo documentos específicos que detallen más sobre la quimioterapia neoadyuvante en el contexto del osteosarcoma o el sarcoma de Ewing pediátrico.


In [13]:
print(f"Fragmentos recuperados inicialmente: 8")
print(f"Fragmentos que pasaron el filtro de relevancia: {len(resultado['docs'])}")
for d in resultado['docs']:
    print(f"  - [{d.metadata['enfermedad']}] {d.metadata['title']}: {d.page_content[:100]}...")

Fragmentos recuperados inicialmente: 8
Fragmentos que pasaron el filtro de relevancia: 1
  - [general] Chemotherapy to Treat Cancer: How chemotherapy is used with other cancer treatments
When used with other treatments, chemotherapy ...


In [ ]:
# Se ven los 8 fragmentos recuperados antes del filtrado
docs_sin_filtrar = qdrant.similarity_search("¿Qué es la quimioterapia neoadyuvante?", k=8)

print("---- Los 8 fragmentos recuperados ----")
for i, d in enumerate(docs_sin_filtrar):
    print(f"\n[{i+1}] [{d.metadata['enfermedad']}] {d.metadata['title']}")
    print(d.page_content[:300])

print("\n\n---- Contenido COMPLETO del único fragmento que pasó el filtro ----")
print(resultado['docs'][0].page_content)

---- Los 8 fragmentos recuperados por el retriever ----

[1] [osteosarcoma] Osteosarcoma Treatment (PDQ®)–Patient Version
Chemotherapy
Chemotherapy (also called chemo) uses drugs to stop the growth of cancer cells. Chemotherapy either kills the cancer cells or stops them from dividing.   Chemotherapy may be given alone or with other types of treatment.
Chemotherapy for osteosarcoma and UPS isinjectedinto a vein. When g

[2] [general] Chemotherapy to Treat Cancer
Chemotherapy works against cancer by killing fast-growing cancer cells.
Credit: National Cancer Institute
Chemotherapy(also called chemo) is a type of cancer treatment that uses drugs to kill cancercells.
On This Page
How chemotherapy works against cancer
How chemotherapy works against cancer
Which 

[3] [osteosarcoma] Osteosarcoma Treatment (PDQ®)–Patient Version
For tumors  that have recurred twice, treatment may include:surgery to remove the cancer and chemotherapychemotherapy alone
surgery to remove the cancer and chemother

### Verificación de fidelidad a los documentos (caso: quimioterapia neoadyuvante)

A la pregunta "¿Qué es la quimioterapia neoadyuvante?", el retriever recupera 8 fragmentos, de los cuales solo 1, el cual define explícitamente "neoadjuvant chemotherapy", pasa el filtro de relevancia, ya que el resto solo habla de quimioterapia en general sin mencionar ese término concreto. La respuesta generada se contrasta frase a frase con el fragmento retenido, confirmando que el modelo no añade información que no esté presente en el documento. El sistema también reconoce que no dispone de información específica sobre neoadyuvancia en osteosarcoma/Ewing en el contexto actual, en vez de inventar una respuesta más completa de la que tiene soporte, lo que supone un comportamiento esperado y deseable para un sistema de apoyo clínico.

> **Nota:** el grafo de esta sección usa una versión mínima de `RAGState` para validar inicialmente que el pipeline básico de retrieval y generación funciona correctamente. En la Sección 7 se amplía `RAGState` y se reconstruye el grafo para incorporar los perfiles de consulta y la integración con el modelo predictivo.

## 6) Integración con el modelo predictivo

Se carga el modelo XGBoost entrenado en `03_modelo_ML.ipynb` junto con el esquema exacto de categorías que usó durante el entrenamiento, para poder construir pacientes nuevos sin riesgo de que un orden de categorías distinto produzca predicciones incorrectas.

In [16]:
with open('../models/xgboost_final.pkl', 'rb') as f:
    modelo_xgb = pickle.load(f)

with open('../models/esquema_categorias.json', 'r', encoding='utf-8') as f:
    esquema = json.load(f)

# Umbral de decisión para β=1.5, como se decidió en el 03_modelo_ML.ipynb
UMBRAL_DECISION = 0.18  

print("Modelo y esquema cargados correctamente")
print(f"Variables categóricas: {list(esquema['categoricas'].keys())}")
print(f"Variables numéricas: {list(esquema['numericas'].keys())}")

Modelo y esquema cargados correctamente
Variables categóricas: ['age_group', 'sex', 'tumor_type', 'primary_site', 'stage', 'surgery_code', 'radiation', 'chemotherapy']
Variables numéricas: ['year_diagnosis']


In [19]:
def construir_paciente(datos: dict) -> pd.DataFrame:
    fila = {}
    for col, categorias in esquema['categoricas'].items():
        valor = datos.get(col)
        if valor is not None and valor not in categorias:
            raise ValueError(f"Valor '{valor}' no es válido para '{col}'. Opciones: {categorias}")
        fila[col] = pd.Categorical([valor], categories=categorias)

    for col, rango in esquema['numericas'].items():
        valor = datos.get(col)
        fila[col] = [valor]
        if valor is not None:
            if valor < rango['min']:
                print(f" AVISO!!!! {col}={valor} Está por debajo del rango de entrenamiento "
                      f"(mínimo visto: {rango['min']}) — revisa si es un error de escritura.")
            elif valor > rango['max']:
                print(f"OJO!!! {col}={valor} Es posterior al dato más reciente usado para "
                      f"entrenar el modelo ({rango['max']}). Es el caso de uso esperado para "
                      f"pacientes actuales, pero la predicción extrapola la tendencia aprendida "
                      f"hasta {rango['max']} sin conocer cambios y avances en los tratamientos.")

    df_paciente = pd.DataFrame(fila)
    orden_entrenamiento = list(esquema['numericas'].keys()) + list(esquema['categoricas'].keys())
    
    return df_paciente[orden_entrenamiento]


def predecir_paciente(datos: dict) -> dict:
    """Devuelve la predicción del modelo para un paciente, con contexto para el RAG."""
    X_paciente = construir_paciente(datos)
    proba = modelo_xgb.predict_proba(X_paciente)[0, 1]
    prediccion = int(proba >= UMBRAL_DECISION)

    return {
        "probabilidad_no_supervivencia_5a": round(float(proba), 3),
        "prediccion_riesgo_alto": bool(prediccion),
        "umbral_usado": UMBRAL_DECISION,
        "datos_paciente": datos,
    }

In [20]:
paciente_ejemplo = {
    "age_group": "10-14 years",
    "sex": "Female",
    "year_diagnosis": 2022,
    "tumor_type": "Ewing sarcoma",
    "primary_site": "Pelvis / hip",
    "stage": "Distant",
    "surgery_code": "No surgery",
    "radiation": "Radiation",
    "chemotherapy": "Yes",
}

resultado_prediccion = predecir_paciente(paciente_ejemplo)
print(resultado_prediccion)

{'probabilidad_no_supervivencia_5a': 0.224, 'prediccion_riesgo_alto': True, 'umbral_usado': 0.18, 'datos_paciente': {'age_group': '10-14 years', 'sex': 'Female', 'year_diagnosis': 2022, 'tumor_type': 'Ewing sarcoma', 'primary_site': 'Pelvis / hip', 'stage': 'Distant', 'surgery_code': 'No surgery', 'radiation': 'Radiation', 'chemotherapy': 'Yes'}}


## 7) Perfiles de consulta: oncólogo vs familia

Se amplía el estado del grafo para incluir el perfil de quien pregunta y, opcionalmente, la predicción del modelo para un paciente concreto. Por seguridad clínica, al perfil familia nunca se le pasa la cifra exacta de probabilidad en el prompt, solo una valoración del resultado, para que sea estructuralmente imposible que el modelo la revele.

### Glosario de variables para el modelo

Se detectó durante las pruebas (caso "stage desconocido") que el LLM puede interpretar de forma incorrecta el significado de una variable ausente si no se le da contexto explícito. Por ejemplo, confundió "estadio no registrado" con "diagnóstico no confirmado", dos cosas distintas. En vez de arreglar ese caso concreto, se añade un glosario fijo con el significado de cada variable del modelo, incluido al prompt del sistema junto con los documentos y la predicción. Así se previene el mismo tipo de ambigüedad para cualquier variable, no solo la que causó el error observado.

In [33]:
GLOSARIO_VARIABLES = """
Glosario de las variables usadas por el modelo predictivo (para tu interpretación, no lo repitas
literalmente al usuario salvo que pregunte específicamente por el significado de una variable):

- age_group: grupo de edad del paciente al diagnóstico.
- sex: sexo del paciente.
- tumor_type: subtipo histológico del tumor (Ewing sarcoma u Osteosarcoma).
- primary_site: localización anatómica donde se originó el tumor.
- stage: estadio de extensión del tumor (Localized/Regional/Distant). Si aparece como ausente
  o no especificado, significa que el registro histórico no recogió ese dato (por ejemplo, por
  el año de diagnóstico del paciente) — NO significa que el diagnóstico del tumor sea incierto
  o esté sin confirmar. El diagnóstico del tipo de tumor es independiente de si se registró su
  estadio.
- surgery_code: tipo de cirugía realizada (o si no se realizó cirugía).
- radiation: si el paciente recibió radioterapia.
- chemotherapy: si el paciente recibió quimioterapia.
- year_diagnosis: año en que se diagnosticó el tumor.

Estas variables proceden de un registro poblacional (SEER), no de una historia clínica
completa: no incluyen biopsias, pruebas de imagen, marcadores moleculares ni comorbilidades.
"""

In [ ]:
class RAGState(TypedDict):
    question: str
    perfil: str  # "oncologo" o "familia"
    prediccion: Optional[dict]
    docs: list
    answer: str

def formatear_contexto_prediccion(prediccion: dict, perfil: str) -> str:
    if not prediccion:
        return ""
    p = prediccion

    # Perfil del oncólogo/médico
    if perfil == "oncologo":
        return f"""
        PREDICCIÓN DEL MODELO PARA ESTE PACIENTE CONCRETO (dato obligatorio a mencionar en tu
        respuesta si la pregunta trata sobre pronóstico, riesgo o supervivencia):
        - Probabilidad estimada de no supervivencia a 5 años: {p['probabilidad_no_supervivencia_5a']*100:.1f}%
        - Clasificación de riesgo: {'Alto' if p['prediccion_riesgo_alto'] else 'No alto'} (umbral de decisión: {p['umbral_usado']})
        - Datos del paciente usados: {p['datos_paciente']}
        Es una estimación de apoyo a la decisión clínica basada en datos históricos de registro
        (SEER), no un diagnóstico ni una certeza.
        """

    # Perfil familia del paciente
    if p['prediccion_riesgo_alto']:
        return """
        DATOS DEL PACIENTE: el modelo identifica varios factores que requieren especial
        atención y seguimiento cercano por parte del equipo médico. No dispones de ninguna
        cifra ni porcentaje, no los menciones ni los inventes. Sé cercano y honesto sin
        generar alarma: reconoce que conviene un seguimiento atento, transmite que el equipo
        médico está para acompañar a la familia en cada paso, y anímales explícitamente a
        hablar con el oncólogo sobre el pronóstico y las opciones de tratamiento, que es quien
        debe dar esa información con el contexto clínico completo que tú no tienes.
        """
    else:
        return """
        DATOS DEL PACIENTE: con la información disponible, el modelo no identifica factores
        de alto riesgo. No dispones de ninguna cifra ni porcentaje, no los menciones ni los
        inventes. Puedes transmitir esto de forma esperanzadora y cercana, recordando siempre
        que el seguimiento del equipo médico sigue siendo la referencia principal.
        """

In [34]:
def generate_node(state: RAGState) -> RAGState:
    contexto_prediccion = formatear_contexto_prediccion(state.get("prediccion"), state["perfil"])

    if state["perfil"] == "oncologo":
        tono = """
        Responde como un asistente clínico dirigido a un oncólogo pediátrico. Puedes usar
        terminología médica sin simplificarla. Si hay predicción del modelo disponible,
        indica la probabilidad exacta y el umbral de decisión usado, y recuerda brevemente
        que es una estimación de apoyo, no un diagnóstico, basada en datos de registro
        poblacional (SEER) sin variables clínicas finas (biopsia, imagen, comorbilidades).
        """
    else:
        tono = """
        Responde como asistente dirigido a la familia de un paciente pediátrico que no cuentan con
        conocimientos médicos previos. Usa lenguaje sencillo y empático, evita jerga médica
        sin explicarla. Nunca menciones cifras o porcentajes de riesgo, aunque los conocieras, ya que 
        no los tienes disponibles para este perfil.
        """

    inputs = {"messages": [
        ("system", f"""
        Eres un asistente que responde preguntas sobre osteosarcoma y sarcoma de Ewing
        pediátrico. Tienes estas fuentes de información:

        {GLOSARIO_VARIABLES}

        FUENTE 1 — Documentos de referencia general:
        ----------
        {state['docs']}
        --------------

        FUENTE 2 — Datos específicos del paciente actual (si están presentes, úsalos):
        {contexto_prediccion if contexto_prediccion else "No se han proporcionado datos de un paciente concreto en esta consulta."}

        {tono}
        """),
        ("human", f"""{state['question']}
        1. Responde en español, de forma clara.
        2. Si la pregunta trata sobre pronóstico/riesgo/supervivencia y tienes datos del
           paciente (FUENTE 2), inclúyelos siempre en tu respuesta.
        3. Si no ves documentos relevantes en la FUENTE 1 para el resto de la pregunta,
           dilo explícitamente en vez de inventar información.
        4. Nunca presentes la predicción del modelo como un diagnóstico definitivo.""")
    ]}
    messages_langchain = convert_to_openai_messages(inputs['messages'])
    resp = llm.invoke(messages_langchain).content
    state['answer'] = resp
    return state

In [35]:
graph = StateGraph(RAGState)
graph.add_node("retrieve", retrieve_node)
graph.add_node("postfiltering", postfiltering_node)
graph.add_node("generate", generate_node)
graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "postfiltering")
graph.add_edge("postfiltering", "generate")
graph.add_edge("generate", END)

rag = graph.compile()
print("Grafo actualizado con perfiles y seguridad clínica en la predicción")

Grafo actualizado con perfiles y seguridad clínica en la predicción


In [36]:
def seleccionar_perfil() -> str:
    while True:
        resp = input("¿Eres oncólogo/a o familiar del paciente? (oncologo/familia): ").strip().lower()
        if resp in ["oncologo", "familia"]:
            return resp
        print("Opción no válida, escribe 'oncologo' o 'familia'.")


def recoger_datos_paciente_guiado() -> dict:
    """Recoge los datos de un paciente campo a campo, validando contra el esquema del modelo."""
    print("\nIntroduce los datos del paciente (elige siempre una opción de la lista):\n")
    datos = {}
    for col, categorias in esquema['categoricas'].items():
        while True:
            print(f"{col} — opciones: {categorias}")
            valor = input(f"  > ").strip()
            if valor in categorias:
                datos[col] = valor
                break
            print("  Valor no válido, elige exactamente una de las opciones mostradas.\n")

    for col, rango in esquema['numericas'].items():
        while True:
            valor_str = input(f"{col} (numérico, valores vistos en entrenamiento: {rango['min']}-{rango['max']}): ").strip()
            try:
                datos[col] = int(valor_str)
                break
            except ValueError:
                print("  Introduce un número válido.\n")

    return datos

In [31]:
perfil_actual = seleccionar_perfil()

usar_paciente = input("¿Quieres introducir un paciente para predicción? (s/n): ").strip().lower()
prediccion_actual = None
if usar_paciente == "s":
    datos_paciente = recoger_datos_paciente_guiado()
    prediccion_actual = predecir_paciente(datos_paciente)
    print(f"\nPredicción calculada: {prediccion_actual}\n")

pregunta = input("\n¿Qué quieres preguntar?: ")
resultado = rag.invoke({"question": pregunta, "perfil": perfil_actual, "prediccion": prediccion_actual})
print("\n--- Respuesta ---")
print(resultado["answer"])

Opción no válida, escribe 'oncologo' o 'familia'.

Introduce los datos del paciente (elige siempre una opción de la lista):

age_group — opciones: ['00 years', '01-04 years', '05-09 years', '10-14 years', '15-19 years']
sex — opciones: ['Female', 'Male']
tumor_type — opciones: ['Ewing sarcoma', 'Osteosarcoma (other subtypes)', 'Osteosarcoma NOS']
primary_site — opciones: ['CNS / meninges', 'Lower limb bones', 'Other / NOS', 'Pelvis / hip', 'Skull / head / jaw', 'Trunk / ribs / sternum', 'Upper limb bones', 'Vertebral column']
stage — opciones: ['Localized', 'Regional', 'Distant']
surgery_code — opciones: ['No surgery', 'Partial resection', 'Radical resection', 'Surgery NOS / Unknown']
radiation — opciones: ['No radiation', 'Radiation', 'Recommended / Unknown', 'Refused']
chemotherapy — opciones: ['No/Unknown', 'Yes']

Predicción calculada: {'probabilidad_no_supervivencia_5a': 0.077, 'prediccion_riesgo_alto': False, 'umbral_usado': 0.18, 'datos_paciente': {'age_group': '15-19 years', 's

### Verificación del comportamiento por perfil

Durante el desarrollo se probó el flujo completo con dos casos opuestos: 

- Un paciente con factores de mal pronóstico (`stage=Distant`, `primary_site=Pelvis/hip`, sin cirugía → probabilidad 22.4%, riesgo alto)
- Un paciente con factores de buen pronóstico (`stage=Localized`, cirugía radical, con quimioterapia → probabilidad 7.7%, riesgo no alto). 

El objetivo era confirmar que el sistema discrimina correctamente entre ambos escenarios y que el perfil familia nunca revela cifras, independientemente de si el pronóstico es favorable o no.

En ambos casos, el perfil familia respondió sin mencionar ningún porcentaje, con tono cercano y empático, y redirigiendo explícitamente al equipo médico para cualquier duda sobre el pronóstico. El perfil oncólogo (probado en el caso de mal pronóstico) sí incluyó la cifra exacta y el umbral de decisión, junto con el aviso de que es una estimación de apoyo, no un diagnóstico.

**Nota:** La salida que queda guardada en el notebook corresponde solo a la última ejecución, no a los dos casos descritos arriba.

## 8) Evaluación cualitativa

Se prueba el sistema de forma sistemática y reproducible (sin `input()`), combinando:
- **3 perfiles de paciente sintéticos**: uno con buen pronóstico, otro con mal pronóstico, y otro  con `stage` desconocido para verificar que el modelo maneja bien el `NaN` nativo también en este contexto.
- **1-2 pacientes reales del test set**: con desenlace ya conocido, pero usados aquí como caso de estudio ilustrativo de la calidad de la explicación, no para validar la precisión del modelo.
- **Una pregunta fuera del alcance de los documentos**: para confirmar que el sistema no inventa información.

Cada caso se prueba con ambos perfiles de consulta (oncólogo / familia).

In [37]:
# Perfiles sintéticos
paciente_mal_pronostico = {
    "age_group": "10-14 years", "sex": "Female", "year_diagnosis": 2022,
    "tumor_type": "Ewing sarcoma", "primary_site": "Pelvis / hip", "stage": "Distant",
    "surgery_code": "No surgery", "radiation": "Radiation", "chemotherapy": "Yes",
}

paciente_buen_pronostico = {
    "age_group": "15-19 years", "sex": "Male", "year_diagnosis": 2021,
    "tumor_type": "Osteosarcoma NOS", "primary_site": "Lower limb bones", "stage": "Localized",
    "surgery_code": "Radical resection", "radiation": "No radiation", "chemotherapy": "Yes",
}

paciente_stage_desconocido = {
    "age_group": "05-09 years", "sex": "Male", "year_diagnosis": 2002,
    "tumor_type": "Osteosarcoma NOS", "primary_site": "Upper limb bones",
    # 'stage': se salta a propósito
    "surgery_code": "Radical resection", "radiation": "No radiation", "chemotherapy": "Yes",
}

# Pacientes reales del test set
df_test_completo = pd.read_csv('../data/splits/test.csv')

paciente_real_evento = df_test_completo[df_test_completo['target'] == 1].iloc[0]
paciente_real_supervive = df_test_completo[df_test_completo['target'] == 0].iloc[0]

print("Paciente real (evento=1, no sobrevive a 5 años):")
print(paciente_real_evento.drop('target').to_dict())
print(f"\nPaciente real (evento=0, sobrevive):")
print(paciente_real_supervive.drop('target').to_dict())

casos_prueba = [
    {"nombre": "Mal pronóstico", "paciente": paciente_mal_pronostico,
     "pregunta": "¿Cuál es el pronóstico y qué tratamiento se recomienda?"},
    {"nombre": "Buen pronóstico", "paciente": paciente_buen_pronostico,
     "pregunta": "¿Cuál es el pronóstico y qué tratamiento se recomienda?"},
    {"nombre": "Stage desconocido", "paciente": paciente_stage_desconocido,
     "pregunta": "¿Qué implica no tener el estado confirmado en el pronóstico?"},
    {"nombre": "Paciente real (evento)", "paciente": paciente_real_evento.drop('target').to_dict(),
     "pregunta": "¿Cuál es el pronóstico de este paciente?"},
    {"nombre": "Paciente real (vive)", "paciente": paciente_real_supervive.drop('target').to_dict(),
     "pregunta": "¿Cuál es el pronóstico de este paciente?"},
    {"nombre": "Fuera de alcance", "paciente": None,
     "pregunta": "¿Qué tratamiento es mejor para el cáncer de mama en una mujer adulta?"},
]

resultados_evaluacion = []

for caso in casos_prueba:
    prediccion = predecir_paciente(caso["paciente"]) if caso["paciente"] else None
    for perfil in ["oncologo", "familia"]:
        resultado = rag.invoke({"question": caso["pregunta"], "perfil": perfil, "prediccion": prediccion})
        resultados_evaluacion.append({
            "caso": caso["nombre"], "perfil": perfil, "pregunta": caso["pregunta"],
            "prediccion": prediccion, "respuesta": resultado["answer"],
        })
        print(f"\n{'='*70}\n[{caso['nombre']} — {perfil}]\n{'='*70}")
        print(f"Pregunta: {caso['pregunta']}")
        if prediccion:
            print(f"Predicción: {prediccion['probabilidad_no_supervivencia_5a']*100:.1f}% (riesgo alto: {prediccion['prediccion_riesgo_alto']})")
        print(f"\nRespuesta:\n{resultado['answer']}")

Paciente real (evento=1, no sobrevive a 5 años):
{'year_diagnosis': 2009, 'age_group': '15-19 years', 'sex': 'Female', 'tumor_type': 'Ewing sarcoma', 'primary_site': 'Pelvis / hip', 'stage': 'Distant', 'surgery_code': 'No surgery', 'radiation': 'Radiation', 'chemotherapy': 'Yes'}

Paciente real (evento=0, sobrevive):
{'year_diagnosis': 2009, 'age_group': '15-19 years', 'sex': 'Male', 'tumor_type': 'Osteosarcoma (other subtypes)', 'primary_site': 'Lower limb bones', 'stage': 'Localized', 'surgery_code': 'Radical resection', 'radiation': 'No radiation', 'chemotherapy': 'No/Unknown'}

[Mal pronóstico — oncologo]
Pregunta: ¿Cuál es el pronóstico y qué tratamiento se recomienda?
Predicción: 22.4% (riesgo alto: True)

Respuesta:
El pronóstico para el paciente en cuestión, que es una niña de 10 a 14 años diagnosticada con sarcoma de Ewing en 2022, es el siguiente:

- **Probabilidad estimada de no supervivencia a 5 años**: 22.4%
- **Clasificación de riesgo**: Alto (umbral de decisión: 0.18)

E

### Interpretación de la evaluación cualitativa

Se han probado 6 casos × 2 perfiles = 12 combinaciones, cubriendo pacientes sintéticos (buen/mal pronóstico, `stage` ausente), pacientes reales del test set con desenlace conocido, y una
pregunta fuera del alcance del sistema.

**Seguridad clínica en el perfil familiar:** en los 6 casos, cero cifras o porcentajes revelados, tono cercano sin alarmismo, y redirección explícita al oncólogo en todos ellos. Funciona tal
como se diseñó en la Sección 7.

**Consistencia en los casos reales:** el paciente real con evento (`target=1`) recibe 64.1% de riesgo, mientras que el paciente real superviviente (`target=0`) recibe 14.8%. Ambos coherentes con su desenlace real, aportando un caso de estudio ilustrativo de calidad de explicación.

**Corrección de un error detectado en las pruebas:** el caso `stage` ausente inicialmente generó una interpretación incorrecta, ya que el LLM confundió "estadio no registrado" con "diagnóstico no confirmado", dos conceptos que son distintos. Se ha corregido añadiendo un `GLOSARIO_VARIABLES` al prompt del sistema (Sección 7), en vez de un parche puntual para ese caso concreto, porque de esta manera se previene el mismo tipo de ambigüedad para cualquier otra variable. Tras el cambio, ambos perfiles interpretan correctamente la ausencia de `stage` como una limitación del registro histórico, no del diagnóstico.

**Casos fuera de alcance:** el sistema reconoce bien cuándo una pregunta no corresponde a su ámbito y no esta relacionada con el osteosarcoma o Ewing pediátrico. No inventa información y sigue un comportamiento correcto.

El sistema, por tanto, combina de forma fiable las dos fuentes de información, que son los documentos y la predicción del modelo. Además mantiene la seguridad clínica diseñada para el perfil familia y el único fallo real detectado en las pruebas, la interpretación de `stage` ausente, se corrigió.

## 9) Cierre y preparación para la API

Se centralizan aquí los parámetros de configuración usados en el notebook para que la API de FastAPI los reutilice sin duplicar valores repartidos por el código. La lógica del grafo se extraerá a un módulo Python compartido (`src/rag_pipeline.py`) al construir la API, para que notebook y API reutilicen exactamente el mismo código en vez de mantener dos copias que puedan desincronizarse.

In [38]:
CONFIG_RAG = {
    "qdrant_collection": COLLECTION_NAME,
    "qdrant_host": "localhost",
    "qdrant_port": 6333,
    "embedding_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "embedding_dim": 384,
    "llm_model": "gpt-4o-mini",
    "retrieval_k": 8,
    "modelo_xgb_path": "../models/xgboost_final.pkl",
    "esquema_categorias_path": "../models/esquema_categorias.json",
    "umbral_decision": UMBRAL_DECISION,
}

with open('../models/config_rag.json', 'w', encoding='utf-8') as f:
    json.dump(CONFIG_RAG, f, ensure_ascii=False, indent=2)

print("Configuración guardada en ../models/config_rag.json")
print(json.dumps(CONFIG_RAG, ensure_ascii=False, indent=2))

Configuración guardada en ../models/config_rag.json
{
  "qdrant_collection": "tfm_oncologia_pediatrica",
  "qdrant_host": "localhost",
  "qdrant_port": 6333,
  "embedding_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
  "embedding_dim": 384,
  "llm_model": "gpt-4o-mini",
  "retrieval_k": 8,
  "modelo_xgb_path": "../models/xgboost_final.pkl",
  "esquema_categorias_path": "../models/esquema_categorias.json",
  "umbral_decision": 0.18
}
